# Fri and Merkle Tree

## Merkle Trees

The tree is built from a hash function $H$ applied bottom-up:

$$\text{Level 0:} \quad h_i = H(v_i) \quad \text{for each leaf } i$$
$$\text{Level 1:} \quad h_{2i,\,2i+1} = H(h_{2i} \,\|\, h_{2i+1}) \quad \text{for each adjacent pair}$$
$$\vdots$$
$$\text{Root:} \quad \text{one final hash that binds the entire vector}$$

The key property is that changing any leaf $v_i$ changes the root. The root is a *binding* commitment to the entire vector.

The three operations we will implement are:

| Operation | Input | Output |
|:---|:---|:---|
| `merkle_commitment(leaves)` | list of $N$ field elements | all tree levels (list of lists of hashes) |
| `merkle_open(tree, i)` | tree + leaf index $i$ | sibling hashes along the path to the root |
| `verify_merkle_proof(leaf, proof, root, i)` | leaf, proof, root, index | `True` or `False` |

The root (last element of the returned tree) is what the prover publishes. Everything else stays with the prover until the verifier asks for a specific leaf.

In [4]:
from hashlib import sha3_256
from zk_adventures_types import F

x = F(3)
hash = sha3_256(x.to_bytes()).hexdigest()
print(hash)

y = sha3_256(bytes.fromhex(hash)).hexdigest()
print(y)

bae484b67d98e56926e87f8cd01da5f32d6916125cc0b04bc9b6ebbe02f950b5
baafe263a6de91cf278fb031449742aa6dfbee51e62f65d94c84204fe2195e12


### Building the tree

For a vector of $N = 2^k$ leaves the tree has $k + 1$ levels. At each step we hash adjacent pairs:

```
leaves:  [v₀,   v₁,   v₂,   v₃,   v₄,   v₅,   v₆,   v₇]
level 0: [h₀,   h₁,   h₂,   h₃,   h₄,   h₅,   h₆,   h₇]   ← H(vᵢ.to_bytes())
level 1: [h₀₁,        h₂₃,        h₄₅,        h₆₇      ]   ← H(hₗₑft.encode() + hᵣᵢᵍʰᵗ.encode())
level 2: [h₀₁₂₃,                  h₄₅₆₇                ]
level 3: [root                                           ]
```

Each parent is `sha3_256(left_hex_string.encode() + right_hex_string.encode()).hexdigest()`, where `left` and `right` are the two child hex-digest strings. The root (the single hash at the last level) is the **commitment**.



In [6]:
import math

def merkle_commitment(leaves):
    raise NotImplementedError()
    return merkle_tree

def merkle_open(merkle_tree, index):
    raise NotImplementedError()
    return proof

V2 = [F(i) for i in range(1,17)]
cm2 = merkle_commitment(V2)
for level in cm2:
    print(level)

print(merkle_open(cm2,3))

NotImplementedError: 

In [ ]:

#[TEST]
V1 = [F(i) for i in range(1,3)]
cm1 = merkle_commitment(V1)
assert len(cm1) == 2   
assert cm1[0] == ['c071a92a8fd2403faf107a10ea2b2524296241894869d96a77031e3a804721e4', 'a50457e609c84942d4ce6fa24c8e0588cb274d594505c16f939f060de9970d44']
assert cm1[1] == ['63bc2f5840038acd04e8f6648cfbd8a2f5fc557c6a7438524cc3087eadd6cd8e']

#[TEST]
V2 = [F(i) for i in range(1, 17)]
cm2 = merkle_commitment(V2)

# Tree has log2(16) + 1 = 5 levels
assert len(cm2) == 5
# Each level halves in size
assert len(cm2[0]) == 16
assert len(cm2[1]) == 8
assert len(cm2[4]) == 1
# Known root
assert cm2[-1][0] == '291d7317a5f39b160cb53bf7563f6300ddf51c35bde8a741dcdb2e05ba3dbff6'
print("merkle_commitment tests passed!")
print("Root:", cm2[-1][0])

merkle_commitment tests passed!
Root: 291d7317a5f39b160cb53bf7563f6300ddf51c35bde8a741dcdb2e05ba3dbff6


### Opening a leaf and verifying a proof

To prove that $v_i$ is the leaf at index $i$ of the committed vector, the prover reveals the **sibling hash** at every level on the path from the leaf to the root. This path has length $\log_2 N$. The proof consists of only $\log_2 N$ hashes.

For a leaf at index $i$ the sibling at each level is:
- index $i+1$ if $i$ is even (the node is a left child),
- index $i-1$ if $i$ is odd (the node is a right child).

Then move up: $i \leftarrow \lfloor i/2 \rfloor$ and repeat.

The **verifier** recomputes the path: starting from $H(v_i)$, at each step combine the current hash with the provided sibling.

In [ ]:
def verify_merkle_proof(leaf, proof, root, index):
    raise NotImplementedError()
    return current_hash == root

root = cm2[-1][0]
verify_merkle_proof(V2[3], merkle_open(cm2, 3), cm2[-1][0], 3)


True

In [ ]:
#[TEST]
proof = merkle_open(cm2, 3)
print(proof)
# Proof has one sibling per level (4 levels below root)
assert len(proof) == 4
# Known sibling hashes along the path for index 3
assert proof == [
    'bae484b67d98e56926e87f8cd01da5f32d6916125cc0b04bc9b6ebbe02f950b5',
    '63bc2f5840038acd04e8f6648cfbd8a2f5fc557c6a7438524cc3087eadd6cd8e',
    '19c4741b88185acd7329a711ba02e6aaaa903951441c505b6df6c87fee71573c',
    '775fdddd538de858a9b3ead0d1040222eac0e38fae485d0fbb957900ed1f290b',
]
print("merkle_open tests passed!")

['bae484b67d98e56926e87f8cd01da5f32d6916125cc0b04bc9b6ebbe02f950b5', '63bc2f5840038acd04e8f6648cfbd8a2f5fc557c6a7438524cc3087eadd6cd8e', '19c4741b88185acd7329a711ba02e6aaaa903951441c505b6df6c87fee71573c', '775fdddd538de858a9b3ead0d1040222eac0e38fae485d0fbb957900ed1f290b']
merkle_open tests passed!


In [ ]:
#[TEST]
root = cm2[-1][0]

# Valid proof for index 3
assert verify_merkle_proof(V2[3], merkle_open(cm2, 3), root, 3) == True

# Valid proof for index 0
assert verify_merkle_proof(V2[0], merkle_open(cm2, 0), root, 0) == True

# Wrong leaf at correct index should fail
assert verify_merkle_proof(V2[5], merkle_open(cm2, 3), root, 3) == False

# Larger tree test
V1 = [F(i) for i in range(1, 1025)]
cm1 = merkle_commitment(V1)
root2 = cm1[-1][0]
index = 458
assert verify_merkle_proof(V1[index], merkle_open(cm1, index), root2, index) == True

print("verify_merkle_proof tests passed!")

verify_merkle_proof tests passed!


In [ ]:
V2 = [F(i) for i in range(1, 1025)]
cm2 = merkle_commitment(V2)
index = 458
proof = merkle_open(cm2, index)
root = cm2[-1][0]
print(verify_merkle_proof(V2[index], proof, root, index))

True


In [ ]:
#[TEST]
root = cm2[-1][0]

# Valid proof for index 3
assert verify_merkle_proof(V2[3], merkle_open(cm2, 3), root, 3) == True

# Valid proof for index 0
assert verify_merkle_proof(V2[0], merkle_open(cm2, 0), root, 0) == True

# Wrong leaf at correct index should fail
assert verify_merkle_proof(V2[5], merkle_open(cm2, 3), root, 3) == False

# Larger tree test
V1 = [F(i) for i in range(1, 1025)]
cm1 = merkle_commitment(V1)
root2 = cm1[-1][0]
index = 458
assert verify_merkle_proof(V1[index], merkle_open(cm1, index), root2, index) == True

print("verify_merkle_proof tests passed!")

verify_merkle_proof tests passed!


---

### From Merkle Trees to FRI

Now that we have an implementation of merkle trees, we can go on to the implementation of FRI.

## Extra - FRI - Fast Reed-Solomon IOP of Proximity

FRI lets a prover convince a verifier that a committed vector of evaluations is *close* to a polynomial of low degree, using only $O(\log N)$ communication. It replaces the `NaiveOracle` from the PLONK notebook with a hash-based scheme: the prover commits once with a Merkle root and later answers any query with a single value plus a Merkle proof.

### Domains

FRI works over a multiplicative subgroup $D_0 = \{1, \omega, \omega^2, \dots, \omega^{N-1}\}$. At each folding step the domain is squared — $D_{i+1} = \{d^2 : d \in D_i\}$ — which halves its size. We precompute all $\log_2 N$ of these domains up front.

In [ ]:
import math

generator = F(3)
omega = generator**(2^9)   # subgroup of order 128

M = 128
D0 = [omega**i for i in range(M)]
domains = [D0]

for i in range(7):
    domains.append([d*d for d in domains[-1][:M//(2^(i+1))]])

In [ ]:
from zk_adventures_types import Polynomial

f = Polynomial([F(i) for i in range(16)])
print(f)

15*X^15 + 14*X^14 + 13*X^13 + 12*X^12 + 11*X^11 + 10*X^10 + 9*X^9 + 8*X^8 + 7*X^7 + 6*X^6 + 5*X^5 + 4*X^4 + 3*X^3 + 2*X^2 + X


## Low-Degree Proof

The prover holds a polynomial $f$ of degree $< d$ and wants to convince the verifier of this without sending $f$. The protocol has two phases:

- **Folding phase** — the prover repeatedly halves the degree of $f$ using verifier-supplied challenges, committing to each intermediate polynomial with a Merkle root.
- **Query phase** — the verifier picks random indices and checks that consecutive commitments are consistent with the claimed fold.

In [ ]:
from hashlib import sha3_256


class Transcript:
    def __init__(self, initialization_bytes: bytes):
        raise NotImplementedError("subclass responsibility")

    def append(self, bytes_to_append: bytes):
        raise NotImplementedError("subclass responsibility")

    def sample(self) -> bytes:
        raise NotImplementedError("subclass responsibility")


class Sha3_256Transcript(Transcript):
    def __init__(self, initialization_bytes: bytes):
        self.hasher = sha3_256()
        self.hasher.update(initialization_bytes)

    def append(self, bytes_to_append: bytes):
        self.hasher.update(bytes_to_append)

    def sample(self) -> bytes:
        result = self.hasher.digest()
        self.hasher = sha3_256()
        self.hasher.update(result)
        return result

    def sample_field_element(self) -> F:
        return F(int(self.sample().hex(), 16))

    def sample_index(self, n: int) -> int:
        return int(self.sample().hex(), 16) % n

In [ ]:
def commit_to_polynomial(f, domain):
    evaluations = [f(d) for d in domain]
    return merkle_commitment(evaluations)

In [ ]:
# [TEST]
assert commit_to_polynomial(f, D0)[-1][0] == '647b7f1d0f223e19c664c80170845ab82e547c4fd37aff747f1d59b36bbeffc9'


### Folding Phase

Given a challenge $\beta$, the prover splits $f$ into even- and odd-indexed coefficients:

$$f(X) = f_{\text{even}}(X^2) + X \cdot f_{\text{odd}}(X^2)$$

and sends the folded polynomial $g(X) = f_{\text{even}}(X) + \beta \cdot f_{\text{odd}}(X)$, which has half the degree. This is repeated $\log_2 d$ times. Challenges are derived non-interactively via Fiat-Shamir: each challenge is sampled from the transcript after appending the previous commitment's root.

In [ ]:
def fold_polynomial(f, challenge):
    n = f.degree()//2 + 1
    f_even = Polynomial([f[2*i]     for i in range(n)])
    f_odd  = Polynomial([f[2*i + 1] for i in range(n)])
    return f_even + challenge * f_odd

print(fold_polynomial(f, F(5)))

89*X^7 + 77*X^6 + 65*X^5 + 53*X^4 + 41*X^3 + 29*X^2 + 17*X + 5


In [ ]:
assert(fold_polynomial(Polynomial([F(1),F(2),F(3),F(4)]), F(10)) == Polynomial([21,43]))
assert fold_polynomial(f, F(5)) == Polynomial([5, 17, 29, 41, 53, 65, 77, 89])


In [ ]:
def folding_protocol(f, domains):
    commits      = [commit_to_polynomial(f, domains[0])]
    root_commits = [commits[-1][-1][0]]
    polynomials  = [f]
    challenges   = []

    transcript = Sha3_256Transcript(initialization_bytes=bytes.fromhex("1234"))
    transcript.append(bytes.fromhex(root_commits[-1]))

    number_of_folds = int(math.log2(len(domains[0])))-1
    
    for i in range(number_of_folds):
        challenge = transcript.sample_field_element()
        challenges.append(challenge)

        g = fold_polynomial(polynomials[-1], challenge)
        polynomials.append(g)

        commits.append(commit_to_polynomial(g, domains[i + 1]))
        root_commits.append(commits[-1][-1][0])
        print(root_commits[-1])
        transcript.append(bytes.fromhex(root_commits[-1]))

    return commits, root_commits, polynomials, challenges, transcript

### Query Phase

The verifier samples random indices from the transcript and checks that each fold is consistent. For a domain point $d_i$, the pair $(f(d_i),\, f(-d_i))$ determines $f_{\text{even}}(d_i^2)$ and $f_{\text{odd}}(d_i^2)$ via:

$$f_{\text{even}}(d_i^2) = \frac{f(d_i) + f(-d_i)}{2}, \qquad f_{\text{odd}}(d_i^2) = \frac{f(d_i) - f(-d_i)}{2\,d_i}$$

The verifier then checks $g(d_i^2) = f_{\text{even}}(d_i^2) + \beta \cdot f_{\text{odd}}(d_i^2)$, authenticated by Merkle proofs at both levels.

In [ ]:
def check_fold(d_i, f_d_i, f_minus_d_i, g_d_i_squared, challenge):
    f_even = (f_d_i + f_minus_d_i) / 2
    f_odd  = (f_d_i - f_minus_d_i) / (2 * d_i)
    assert f_even + challenge * f_odd == g_d_i_squared

In [ ]:
def query_protocol(commits, root_commits, polynomials, challenges, domains, transcript):
    number_of_folds = len(challenges)

    f_di_values, f_neg_di_values, g_di_sq_values = [], [], []
    proofs_di, proofs_neg_di, proofs_di_sq       = [], [], []

    for k in range(number_of_folds):
        index     = transcript.sample_index(len(domains[k]) // 2)
        neg_index = index + len(domains[k]) // 2

        d_i     = domains[k][index]
        neg_d_i = domains[k][neg_index]

        f_di     = polynomials[k](d_i)
        f_neg_di = polynomials[k](neg_d_i)
        g_di_sq  = polynomials[k + 1](d_i * d_i)

        f_di_values.append(f_di)
        f_neg_di_values.append(f_neg_di)
        g_di_sq_values.append(g_di_sq)

        proofs_di.append(merkle_open(commits[k], index))
        proofs_neg_di.append(merkle_open(commits[k], neg_index))
        proofs_di_sq.append(merkle_open(commits[k + 1], index))

        check_fold(d_i, f_di, f_neg_di, g_di_sq, challenges[k])

    return f_di_values, f_neg_di_values, g_di_sq_values, proofs_di, proofs_neg_di, proofs_di_sq

In [ ]:
commits, root_commits, polynomials, challenges, transcript = folding_protocol(f, domains)
query_protocol(commits, root_commits, polynomials, challenges, domains, transcript)

ffbb43bb7c24775090f7ceef1ce2cad158405adf4de96922f0f2cc448d6beffb
073e231dc74852e88dda6f6a3a00a4aa42ed46ced0b47b5773730444ed918141
0f881767fae29b3a68b4d2abf2a1d52d80bdd42912d676daed49cd9343f4f6b8
b5f32a9a054d3879d8f2bb9eb9a8919ba8731924738a6f72222f5180b5854402
a996ae3e5dcb9ae5b08e0eb6cd812bd3dd342df6d9dd9fe314b6c3ed224c93a7
1c9ecfb36d07abb21d9997200460e095ac0cf58c48bd6fe55784dc2f90c83cd2


([28136, 39497, 31193, 20713, 22912, 22912],
 [49245, 55726, 59517, 27331, 22912, 22912],
 [22614, 61372, 36615, 22912, 22912, 22912],
 [['abc7f0f8bad04608a4f7da6cb2c94dd834b47be24eba9bf043a87300fddc0e24',
   'a11dba36c91b11925fe23954603b70c191c78e0cc19dbdab9eeedbbf6790f45f',
   '20ef0204001bdcaa2e04c22f12cd47d95197d21595e923f6a112bedfb75b5488',
   'e1aa2ed9d8fc604b3525575e65a20422b39c03a5476fe5daf95d1dc57424836e',
   '28ae9094e055b080ab0d61f4745aa28dfd1d496948031fca39cc50610d114062',
   '9cd3de838a66d058eb4f72b896c9a89c4228223dbdd60804c74e5aa74f49f05d',
   '1ca19b7b6d875fe57a0a0e25b68289fbccbeabfc668df258ee8a98f80d2676a9'],
  ['f50747ae90b1c6305b38dec02ae2adf608b72da86e10e095e33d8c0c0f1bc91b',
   '41bee9f2a2a08065155883ba4dcc96ecffbd0df8a6f289dc8d68367979a71b84',
   '55cf9e7039fe7fea86784e547ee57be31e1cd8b4ee17d09ff1cecbb6ea8b9fa0',
   '31d83f34196ef5fc8157086f374c506c85bc54eed76ae4cc3ed7dc63a89df4cf',
   '3c11a8f15102a3cf0e4d54466c563894d77efed35881b6560d150fe138637009',
   'a7dfa1ad

In [ ]:
def build_proof_for_low_degree(f, domains):
    commits, root_commits, polynomials, challenges, transcript = folding_protocol(f, domains)
    f_di, f_neg_di, g_di_sq, proofs_di, proofs_neg_di, proofs_di_sq = query_protocol(
        commits, root_commits, polynomials, challenges, domains, transcript
    )
    return root_commits, f_di, f_neg_di, g_di_sq, proofs_di, proofs_neg_di, proofs_di_sq

build_proof_for_low_degree(f, domains)

ffbb43bb7c24775090f7ceef1ce2cad158405adf4de96922f0f2cc448d6beffb
073e231dc74852e88dda6f6a3a00a4aa42ed46ced0b47b5773730444ed918141
0f881767fae29b3a68b4d2abf2a1d52d80bdd42912d676daed49cd9343f4f6b8
b5f32a9a054d3879d8f2bb9eb9a8919ba8731924738a6f72222f5180b5854402
a996ae3e5dcb9ae5b08e0eb6cd812bd3dd342df6d9dd9fe314b6c3ed224c93a7
1c9ecfb36d07abb21d9997200460e095ac0cf58c48bd6fe55784dc2f90c83cd2


(['647b7f1d0f223e19c664c80170845ab82e547c4fd37aff747f1d59b36bbeffc9',
  'ffbb43bb7c24775090f7ceef1ce2cad158405adf4de96922f0f2cc448d6beffb',
  '073e231dc74852e88dda6f6a3a00a4aa42ed46ced0b47b5773730444ed918141',
  '0f881767fae29b3a68b4d2abf2a1d52d80bdd42912d676daed49cd9343f4f6b8',
  'b5f32a9a054d3879d8f2bb9eb9a8919ba8731924738a6f72222f5180b5854402',
  'a996ae3e5dcb9ae5b08e0eb6cd812bd3dd342df6d9dd9fe314b6c3ed224c93a7',
  '1c9ecfb36d07abb21d9997200460e095ac0cf58c48bd6fe55784dc2f90c83cd2'],
 [28136, 39497, 31193, 20713, 22912, 22912],
 [49245, 55726, 59517, 27331, 22912, 22912],
 [22614, 61372, 36615, 22912, 22912, 22912],
 [['abc7f0f8bad04608a4f7da6cb2c94dd834b47be24eba9bf043a87300fddc0e24',
   'a11dba36c91b11925fe23954603b70c191c78e0cc19dbdab9eeedbbf6790f45f',
   '20ef0204001bdcaa2e04c22f12cd47d95197d21595e923f6a112bedfb75b5488',
   'e1aa2ed9d8fc604b3525575e65a20422b39c03a5476fe5daf95d1dc57424836e',
   '28ae9094e055b080ab0d61f4745aa28dfd1d496948031fca39cc50610d114062',
   '9cd3de838a66d05

### Exercise: Verifier

The verifier rebuilds its own transcript (using the public root commits) to reproduce the same challenges and query indices, then:
1. checks each Merkle proof against the corresponding root, and
2. calls `check_fold` to verify each folding step.

In [ ]:
def verify_low_degree_proof(
    root_commits,
    f_di_values, f_neg_di_values, g_di_sq_values,
    proofs_di, proofs_neg_di, proofs_di_sq,
    domains,
):
    number_of_folds = len(f_di_values)

    # ── Phase 1: reproduce the folding challenges ────────────────────────────
    # The verifier has the root commits sent by the prover.  It initialises
    # the transcript with the same seed and replays the folding phase to
    # recover every challenge β_k.

    transcript = Sha3_256Transcript(initialization_bytes=bytes.fromhex("1234"))
    transcript.append(bytes.fromhex(root_commits[0]))

    challenges = []
    for k in range(number_of_folds):
        challenge = ...                              # sample β_k from transcript
        challenges.append(challenge)
        transcript.append(...)                       # advance transcript: append root_commits[k+1]

    # ── Phase 2: reproduce the query indices and verify ──────────────────────
    # The transcript now continues from the same state as the prover's after
    # folding, so sampling here gives the same indices.

    for k in range(number_of_folds):
        index     = ...                              # sample_index from the first half of domains[k]
        neg_index = ...                              # antipodal index (second half of the domain)
        d_i       = domains[k][index]

        # Verify that f(d_i) is consistent with the Merkle commitment at round k
        assert verify_merkle_proof(..., proofs_di[k],     root_commits[k],     ...)

        # Verify that f(-d_i) is consistent with the Merkle commitment at round k
        assert verify_merkle_proof(..., proofs_neg_di[k], root_commits[k],     ...)

        # Verify that g(d_i²) is consistent with the Merkle commitment at round k+1
        assert verify_merkle_proof(..., proofs_di_sq[k],  root_commits[k + 1], ...)

        # Check that the fold relation holds at this query point
        check_fold(d_i, f_di_values[k], f_neg_di_values[k], g_di_sq_values[k], challenges[k])

    return True

In [ ]:
# [TEST]
root_commits, f_di, f_neg_di, g_di_sq, proofs_di, proofs_neg_di, proofs_di_sq = build_proof_for_low_degree(f, domains)
assert verify_low_degree_proof(root_commits, f_di, f_neg_di, g_di_sq, proofs_di, proofs_neg_di, proofs_di_sq, domains)
print("verify_low_degree_proof passed!")